# Notebook 4 — Détection d'anomalies sur les logs

## Contexte

Le notebook 3 a montré que les métriques ne détectent pas  
les pannes de type `exception` et `return` — invisibles dans le CPU,  
la mémoire ou la latence.

Ces pannes sont uniquement visibles dans les **logs applicatifs**.  
Ce notebook implémente 3 algorithmes de détection basés sur les logs.

## Approche

Les logs sont des données textuelles non structurées.  
On les transforme en données numériques via :
1. **Comptage de templates** — fréquence des patterns de log
2. **TF-IDF** — importance des mots rares
3. **DeepLog (LSTM)** — séquence anormale de logs

## Algorithmes comparés

| # | Algorithme | Type | Ce qu'il détecte |
|---|-----------|------|-----------------|
| 1 | Comptage de templates | Statistique | Changement de fréquence |
| 2 | TF-IDF + seuil | ML simple | Mots inhabituels |
| 3 | LSTM (DeepLog) | Deep Learning | Séquences anormales |

## Évaluation

Précision · Rappel · F1-score contre le `ground_truth.csv`

## Structure

| Section | Contenu |
|---------|---------|
| 1. Configuration | Imports, chemins, paramètres |
| 2. Exploration | Structure des logs normaux vs anormaux |
| 3. Comptage templates | Baseline statistique |
| 4. TF-IDF | Détection par mots inhabituels |
| 5. LSTM DeepLog | Détection par séquences |
| 6. Comparaison | Tableau récapitulatif |

In [ ]:
import os, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter
warnings.filterwarnings('ignore')

# ── Racine du projet
PROJET    = Path('/home/eunice/Bureau/Train_ticket/Intelligent_observability')
NORMAL    = PROJET / 'data/normal'
ANOMALIES = PROJET / 'data/anomalies'
FIGURES   = PROJET / 'figures/detection_logs'
RESULTS   = PROJET / 'results'
OUTPUT    = PROJET / 'output'

for dossier in [FIGURES]:
    dossier.mkdir(parents=True, exist_ok=True)

# ── Paramètres Train Ticket
DATES_TT  = ['2023-01-29', '2023-01-30']
FAULT_DUR = 3  # minutes

# ── Fonction extraction niveau de log
def extraire_niveau(log_str):
    match = re.search(r'\b(INFO|ERROR|WARN|DEBUG)\b', str(log_str))
    return match.group(1) if match else 'INCONNU'

# ── Vérification
assert NORMAL.exists(),    f"Introuvable : {NORMAL}"
assert ANOMALIES.exists(), f"Introuvable : {ANOMALIES}"
assert (OUTPUT / 'ground_truth.csv').exists(), "ground_truth.csv introuvable"

print("✓ Configuration OK")
print(f"  Projet    : {PROJET}")
print(f"  Normal    : {NORMAL.exists()}")
print(f"  Anomalies : {ANOMALIES.exists()}")
print(f"  Figures   : {FIGURES}")

: 